In [1]:
import os
import pandas as pd
import re

In [2]:
import json

In [3]:
rub_data_xls_file = '../data/SFB-TR103-Creep-data/SX-CREEP-DATA_LWW-RUB_JAN2023.xlsx'

In [4]:
Rub_data = pd.read_excel(rub_data_xls_file, sheet_name=None, header=None)

In [5]:
Rub_data.keys()

dict_keys(['Overview', ' 001-direction', '110-direction', '111-direction'])

In [6]:
ORIENTATION = '001-direction'

In [7]:
TITLE = Rub_data['Overview'][0][0]

In [8]:
LICENCE = Rub_data['Overview'][0].dropna()[[2,3]].sum()

In [9]:
HEAT_TREATMENT = Rub_data['Overview'][0].dropna()[18]

In [10]:
PUBLICATION = Rub_data['Overview'][0].dropna()[21]

In [11]:
PREPARATION = Rub_data['Overview'][0].dropna()[24]

In [12]:
EXPERIMENTS = Rub_data['Overview'][0].dropna()[[27,28]].sum()

In [13]:
ASSOCIATEDPUBS = Rub_data['Overview'][0].dropna().iloc[-6:].sum()

In [14]:
ASSOCIATEDPUBS

"P. Wollgramm, H. Buck, K. Neuking, A.B. Parsa, S. Schuwalow, J. Rogal, R. Drautz, G. Eggeler, On the role of Re in the stress and temperature dependence of creep of Ni-base single Crystal superalloys, Materials Science and Engineering a, 628 (2015) 382-395H. Buck, P. Wollgramm, A.B. Parsa, G. Eggeler, A quantitative metallographic assessment of the evolution of porosity during processing and creep in single crystal Ni-base super alloys, Materialwissenschaft und Werkstofftechnik, 46 (2015) 577-590A.B. Parsa, P. Wollgramm, H. Buck, A. Kostka, C. Somsen, A. Dlouhy, G. Eggeler, Ledges and grooves at gamma/gamma ' interfaces of single crystal superalloys, Acta Materialia, 90 (2015) 105-117P. Wollgramm, D. Bürger, A.B. Parsa,l K. Neuking, G. Eggeler, The effect of stress, temperature and loading direction on the creep behaviour of Ni-base single crystal superalloy miniature creep specimens, Material at High Temperatures, 33 (2016) 346-360X. Wu, P. Wollgramm, C. Somsen, A. Dlouhy, A. Kostka,

In [15]:
PREPARATION

'P. Wollgramm, D. Bürger, A.B. Parsa,l K. Neuking, G. Eggeler, The effect of stress, temperature and loading direction on the creep behaviour of Ni-base single crystal superalloy miniature creep specimens, Material at High Temperatures, 33 (2016) 346-360'

In [16]:
this_sheet = Rub_data[' 001-direction']

In [17]:
this_metadata = this_sheet.iloc[:6,0].dropna()

In [18]:
tables_in_sheet = []

ntables = int((this_sheet.shape[1]-1)/3)

In [19]:
this_metadata

0    [001] TENSILE CREEP DATA PRESENTED AS TIME (IN...
3    TESTS ARE IDENTIFIED BY TEMPERATURE (IN °C) AN...
Name: 0, dtype: object

In [20]:
for itable in range(ntables - 1):

    this_table = this_sheet.iloc[6:,3*itable:3*itable+2].dropna(how='all')
    this_table = this_table.reset_index(drop=True)
    this_table.columns = [0,1] 
    this_metadata =this_table[0][1]#,3*itable:3*itable+2]
    #this_metadata = this_metadata.split(':')
    rupture_time = re.findall('[\d+h]', this_metadata)
    #key = re.findall('\w+\s?\w+:', this_metadata)[0].replace(':','')
    temperature, stress = this_table[0][0].split('/')

    this_annotated_table = {
        'metadata' : {
            'TITLE' : TITLE,
            'Licence': LICENCE,
            'Heat treatment' : HEAT_TREATMENT,
            'Publication' : PUBLICATION,
            'Experimental preparation': PREPARATION,
            'Experimental description' : EXPERIMENTS,
            'Licence' : LICENCE,
            'Rupture time' : rupture_time[0],
            'Temperature' : {'Value' : temperature, 'Unit' : '°C'}, 
            'Stress' : {'Value': stress, 'Unit' : 'MPa' },
            'Monocrystal' : {'Orientation' : ORIENTATION.strip() }
        },
        'data' : {
            'times' : {
                'Value' : this_table[0][2:].to_list(), #.values[2:],
                'Unit' : 's'
            },
            'elongation' : {
                'Value' : this_table[1][2:].to_list() ,# values[2:],
                'Unit' : 'mm'
            }
        }
    }
    tables_in_sheet.append(this_annotated_table)
    #this_rupture_time = get_rupture_time(this_metadata[0])    

In [21]:
for curve in tables_in_sheet:
    file_basename = curve['metadata']['Monocrystal']['Orientation']+'_T='+curve['metadata']['Temperature']['Value']+'_S='+curve['metadata']['Stress']['Value']+'.json'
    file_full_path = os.path.join(os.path.dirname(rub_data_xls_file), file_basename)
    file_full_path
    with open(file_full_path, 'w') as f:
        json.dump(curve, f, indent=4)